In [53]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np
import pickle

In [54]:
### load the trained model, scaler pickle, onehot

model= load_model('model.h5')

## load the encoder and scaler

with open('onehot_encoder_geo.pkl','rb') as file:
  onehot_encoder_geo= pickle.load(file)


with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender= pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler= pickle.load(file)



In [55]:
input_data = {
    'CreditScore': 500,
    'Geography': 'Germany',
    'Gender': 'Female',
    'Age': 55,
    'Tenure': 2,
    'Balance': 120000,
    'NumOfProducts': 1,
    'HasCrCard': 1,
    'IsActiveMember': 0,
    'EstimatedSalary': 50000
}

In [56]:
#one hot encoding Geography

geo_encoded=onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df= pd.DataFrame(geo_encoded, columns= onehot_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df

C:\Users\aksha\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,0.0,1.0,0.0


In [57]:
input_df= pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,500,Germany,Female,55,2,120000,1,1,0,50000


In [58]:
## Encode categorical variables

input_df['Gender']= label_encoder_gender.transform(input_df['Gender'])

input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,500,Germany,0,55,2,120000,1,1,0,50000


In [59]:
## concatination with onehot encoded 

input_df= pd.concat([input_df.drop("Geography", axis=1),geo_encoded_df], axis=1)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,500,0,55,2,120000,1,1,0,50000,0.0,1.0,0.0


In [60]:
##Scaling the input data

input_scaled= scaler.transform(input_df)
input_scaled

array([[-1.55348047, -1.0862028 ,  1.51646518, -1.04946154,  0.68886601,
        -0.9024366 ,  0.64920267, -1.03615311, -0.87591581, -0.9990005 ,
         1.71207591, -0.57138416]])

In [61]:
## Predict churn

prediction= model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step


array([[0.9467292]], dtype=float32)

In [62]:
prediction_proba= prediction[0][0]

print(prediction_proba)

0.9467292


In [63]:
if prediction_proba > 0.5:
    print('likely to churn')
else :
    print('not likely to churn')

likely to churn
